# vscene2d — a tour

VPython's ergonomics, in 2D, running natively in a Jupyter cell. You write the physics;
the canvas, axes, scaling, and redraws take care of themselves.

Two ways to drive an animation:

| | when to use it |
|---|---|
| `while ...: rate(fps)` | identical to VPython. Fine when `dt` is coarse. |
| `scene.run(step, dt=..., fps=30)` | separates the integration step from the frame rate. Use this whenever `dt` is small. |

Two ways to see it:

| | |
|---|---|
| `Scene(mode="live")` | draws frame by frame in a widget while the loop runs. |
| `Scene(mode="record")` + `scene.player()` | runs at full speed, then gives you a scrubbable animation that **survives saving, exporting, and nbviewer**. |

`mode="auto"` (the default) picks `live` in a notebook and `record` everywhere else.

## 1 — Projectile motion

The VPython script, essentially unchanged.

In [1]:
from vscene2d import *
import math

scene = Scene(title="Projectile motion", mode="record")

ground = Segment(start=vector(-2, 0), end=vector(60, 0), color=color.black)
ball   = Ball(pos=vector(0, 0), radius=0.6, color=color.red, make_trail=True)

v0, theta = 30.0, math.radians(55)
vel = vector(v0*math.cos(theta), v0*math.sin(theta))
g   = vector(0, -9.8)

arrow = attach_arrow(ball, "vel", owner=globals(), scale=0.25, color=color.blue)

def step(dt):
    global vel
    vel = vel + g*dt
    ball.pos = ball.pos + vel*dt

scene.run(step, dt=0.001, until=lambda: ball.pos.y < 0)

print(f"range  = {ball.pos.x:6.2f} m   (analytic {v0**2*math.sin(2*theta)/9.8:.2f})")
print(f"flight = {scene.t:6.2f} s   (analytic {2*v0*math.sin(theta)/9.8:.2f})")
scene.player()

range  =  86.29 m   (analytic 86.30)
flight =   5.02 s   (analytic 5.02)


Note `attach_arrow`: the velocity arrow re-reads `vel` every frame, so it can't drift out of
sync with the physics the way a hand-updated arrow does.

`until=` is checked on every *integration* step, not every frame, so the run stops at the
ground crossing to within `dt` rather than to within a frame.

## 2 — Mass on a spring, with a live energy graph

Watching the mass is the demo; watching kinetic and potential energy trade places while the
total stays flat is the *lesson*. `Graph` is a second canvas driven by the same loop.

In [2]:
from vscene2d import *

scene = Scene(title="SHM", width=560, height=240, mode="record",
              center=vector(1.0, 0), range=1.0)

wall  = Segment(start=vector(0, -0.6), end=vector(0, 0.6), color=color.black, lw=3)
mass  = Ball(pos=vector(1.6, 0), radius=0.12, color=color.blue)
spring = Spring(start=vector(0, 0), end=mass.pos, coils=12, amplitude=0.1)

energy = Graph(width=560, height=220, title="Energy", xtitle="t (s)")
ke  = gcurve(color=color.blue,  label="kinetic",   every=20)
pe  = gcurve(color=color.green, label="potential", every=20)
tot = gcurve(color=color.black, label="total",     every=20)

k, m, x0 = 12.0, 0.5, 1.6      # N/m, kg, m
L0 = 1.0                        # natural length
x, v = x0, 0.0

def step(dt):
    global x, v
    a = -k*(x - L0)/m
    v += a*dt                   # symplectic Euler: energy stays bounded
    x += v*dt
    mass.pos    = vector(x, 0)
    spring.end  = mass.pos
    ke.plot(scene.t, 0.5*m*v*v)
    pe.plot(scene.t, 0.5*k*(x - L0)**2)
    tot.plot(scene.t, 0.5*m*v*v + 0.5*k*(x - L0)**2)

scene.run(step, dt=0.0002, duration=4.0, fps=30)
scene.player()

In [ ]:
energy.player()

`dt = 2e-4` is 100× finer than the frame interval. With a bare `rate(30)` loop that would play
back 100× slower than real time; `run()` puts the extra steps *between* frames instead.

`every=20` thins each trace — 20 000 integration steps is more points than a 560-pixel-wide
plot can show, and drawing them all 30 times a second is what makes browser-based plots stutter.

## 3 — Orbits: an integrator comparison

Same initial conditions, two integrators, one plot. This is the demo where the difference
between Euler and symplectic Euler stops being an abstract claim about "energy drift".

In [ ]:
from vscene2d import *

scene = Scene(title="Kepler orbit", width=520, height=520, mode="record",
              center=vector(0, 0), range=1.6)

sun = Ball(pos=vector(0, 0), radius=0.10, color=color.yellow)

GM = 1.0
r0, v0 = vector(1.0, 0.0), vector(0.0, 1.05)

naive  = Ball(pos=r0, radius=0.035, color=color.red,  make_trail=True)
sympl  = Ball(pos=r0, radius=0.035, color=color.blue, make_trail=True)
Label(pos=vector(0, 1.42), text="red: Euler     blue: symplectic Euler", size=12)

rn, vn = r0, v0
rs, vs = r0, v0

def accel(r):
    return r * (-GM / r.mag**3)

def step(dt):
    global rn, vn, rs, vs
    rn, vn = rn + vn*dt, vn + accel(rn)*dt      # explicit Euler: old r, old v
    vs = vs + accel(rs)*dt                       # symplectic: new v, then r
    rs = rs + vs*dt
    naive.pos, sympl.pos = rn, rs

scene.run(step, dt=0.0005, duration=12.0, fps=30)
scene.player()

The red orbit spirals outward; the blue one closes. Identical `dt`, identical forces — the only
difference is which velocity the position update uses.

## The VPython idiom still works

If you have existing VPython code or want the familiar shape, `rate()` is unchanged:

In [3]:
from vscene2d import *

scene = Scene(mode="record", title="rate() loop")
ball = Ball(pos=vector(0, 0), radius=0.3, color=color.red, make_trail=True)

vel, g, dt = vector(12, 16), vector(0, -9.8), 0.01

while ball.pos.y >= 0:
    rate(100)
    ball.pos = ball.pos + vel*dt
    vel = vel + g*dt

scene.player()

## Saving

`scene.player()` is already self-contained HTML with the frame data inlined — it keeps working
after the kernel dies, after **File → Save and Export as → HTML**, and on nbviewer.

For a standalone file (a course website, a link in an assignment):

```python
scene.save_html("orbit.html")
```